# Tarea 3 Inteligencia Artificial - Procesamiento del lenguaje natural

**Integrantes:**
*   Guillermo Gonzales
*   Juan de Dios Godoy
*   Cristobal Salgado
*   Ignacio Vidal


In [1]:
#!pip install spacy #Instalamos spaCy para poder lematizar las oraciones
#!python -m spacy download en_core_web_sm #descargamos el modelo pre entrenado en ingles, puesto que los resumenes son en ingles
#!pip install langdetect #Para determinar los idiomas del dataframe

In [2]:
#Importamos librerias necesarias
import numpy as np
import pandas as pd
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from collections import Counter

import spacy
import en_core_web_sm
from string import punctuation
from spacy.lang.es.stop_words import STOP_WORDS as stop_words_es
from spacy.lang.en.stop_words import STOP_WORDS as stop_words_en
from spacy.lang.fr.stop_words import STOP_WORDS as stop_words_fr

from gensim.models import Word2Vec

from scipy.spatial.distance import cosine

### Importamos la base de datos y realizamos la limpieza necesaria del modelo

In [3]:
#Cargamos el archivo a utilizar
df = pd.read_excel('Corpus-Agro.xlsx', engine='openpyxl')
print("El tamaño de la base de datos es:", df.shape)
print("El tipo de dato de resumen es:", df['Resumen'].dtype)
df.head()

El tamaño de la base de datos es: (100908, 2)
El tipo de dato de resumen es: object


,URL de Documento,Resumen
0,https://faolex.fao.org/docs/pdf/vie78248.pdf,"This Decree provides for the functions, tasks,..."
1,https://faolex.fao.org/docs/pdf/vie95279.pdf,This Decision approves the Scheme on the devel...
2,https://faolex.fao.org/docs/pdf/ita121539.pdf,This Regional Act sets out the legislative fra...
3,https://faolex.fao.org/docs/pdf/cro126979.pdf,This Regulation amends various provisions of t...
4,https://faolex.fao.org/docs/pdf/chn137120C.pdf...,This Law is enacted for the purposes of guaran...


In [4]:
for i in range(5):
  print(df.iloc[i,1])
  print("****")

This Decree provides for the functions, tasks, powers and organizational structure of the Ministry of Agriculture and Rural Development. The Ministry of Agricultural and Rural Development is a governmental agency which shall perform function of state management in the domains of agriculture, plant protection, animal health, food quality, forestry, salt-making, fisheries, irrigation, rural development, etc. Tasks and powers of the Ministry are specified in the text. 
****
This Decision approves the Scheme on the development of agricultural and forest plant varieties, livestock breeds and aquatic strains. The main objective of the Scheme is to raise the capacity of the system of research, selection, creation, transfer, production and supply of cultivation plant varieties, livestock breeds, forest tree varieties and aquatic strains in order to rapidly raise the yield, quality, competitiveness and effectiveness of agricultural production, forestry and fisheries. The Decision further provid

In [5]:
#Truncamos a 1000 datos
df = df.sample(n=100, random_state=40).reset_index(drop=True)
df.head()

,URL de Documento,Resumen
0,https://faolex.fao.org/docs/pdf/ita32358.pdf,"The Regional Act shall apply to animal fats, s..."
1,https://faolex.fao.org/docs/pdf/ita77436.pdf,This Decree lays down measures to prevent the ...
2,https://faolex.fao.org/docs/pdf/per113114.pdf,La presente Resolución reconoce y aprueba a la...
3,https://faolex.fao.org/docs/pdf/ins213730.pdf,"This Ministerial Regulation, based on the prov..."
4,https://faolex.fao.org/docs/pdf/ken126417.pdf,"These Rules, made under the Animal Diseases Ac..."


In [6]:
# Inicializar un contador para almacenar la cantidad de resúmenes por idioma
idioma_contador = Counter()

# Iterar por cada resumen en la segunda columna
for resumen in df.iloc[:, 1]:
    try:
        # Asegurarse de que el resumen sea un texto (string)
        if isinstance(resumen, str):
            # Detectar el idioma del resumen
            idioma = detect(resumen)
            # Incrementar el contador para ese idioma
            idioma_contador[idioma] += 1
        else:
            # Contar como "indefinido" si no es un texto
            idioma_contador["indefinido"] += 1
    except LangDetectException:
        # Si hay algún error al detectar, también se cuenta como "indefinido"
        idioma_contador["indefinido"] += 1

# Mostrar la cantidad de resúmenes en cada idioma
print("Cantidad de resúmenes por idioma:")
for idioma, cantidad in idioma_contador.items():
    print(f"{idioma}: {cantidad}")



Cantidad de resúmenes por idioma:
en: 62
es: 24
fr: 14


In [7]:
# Asegúrate de que los resultados sean consistentes en cada ejecución
DetectorFactory.seed = 0

# Función para detectar el idioma
def detectar_idioma(resumen):
    if isinstance(resumen, str) and resumen.strip():  # Comprobar si es un string no vacío
        try:
            # Detectar el idioma del resumen
            return detect(resumen)
        except Exception:
            return "Desconocido"  # Si hay un error, devuelve "Desconocido"
    else:
        return "Desconocido"  # Si el resumen no es válido, devuelve "Desconocido"

# Aplicar la función para crear una nueva columna de idioma
df['Idioma'] = df['Resumen'].apply(detectar_idioma)

# Mostrar el DataFrame con la nueva columna
print(df[['Resumen', 'Idioma']].head())


                                             Resumen Idioma
0  The Regional Act shall apply to animal fats, s...     en
1  This Decree lays down measures to prevent the ...     en
2  La presente Resolución reconoce y aprueba a la...     es
3  This Ministerial Regulation, based on the prov...     en
4  These Rules, made under the Animal Diseases Ac...     en


In [8]:
# Función para lematizar el texto según el idioma
def Lematizar(oracion, idioma):
    oracion = str(oracion)
    if idioma == 'en':
        doc = nlp_en(oracion)
    elif idioma == 'es':
        doc = nlp_es(oracion)
    elif idioma == 'fr':
        doc = nlp_fr(oracion)
    else:
        return oracion
    
    # Lematización, pasando a minúsculas y eliminando stopwords y puntuación
    lemas = [
        token.lemma_.lower().strip() for token in doc
        if token.text not in stop_words[idioma] and token.text not in punctuations
    ]
    return " ".join(lemas)


In [9]:
# Cargar los modelos de spaCy para los tres idiomas
nlp_en = spacy.load('en_core_web_sm')
nlp_es = spacy.load('es_core_news_sm')
nlp_fr = spacy.load('fr_core_news_sm')

# Definir stopwords y puntuación
stop_words = {
    'en': set(stop_words_en),
    'es': set(stop_words_es),
    'fr': set(stop_words_fr),
}
punctuations = set(punctuation)

# Aplicar la función a cada resumen
df['Resumen_lem'] = df.apply(lambda row: Lematizar(row['Resumen'], row['Idioma']), axis=1)

In [10]:
df.head()

,URL de Documento,Resumen,Idioma,Resumen_lem
0,https://faolex.fao.org/docs/pdf/ita32358.pdf,"The Regional Act shall apply to animal fats, s...",en,the regional act shall apply animal fat slaugh...
1,https://faolex.fao.org/docs/pdf/ita77436.pdf,This Decree lays down measures to prevent the ...,en,this decree lay measure prevent outbreak sprea...
2,https://faolex.fao.org/docs/pdf/per113114.pdf,La presente Resolución reconoce y aprueba a la...,es,el presente resolución reconocer aprobar briga...
3,https://faolex.fao.org/docs/pdf/ins213730.pdf,"This Ministerial Regulation, based on the prov...",en,this ministerial regulation base provision gov...
4,https://faolex.fao.org/docs/pdf/ken126417.pdf,"These Rules, made under the Animal Diseases Ac...",en,these rules animal diseases act regulate impor...


In [11]:
print(df['Resumen'][4])

These Rules, made under the Animal Diseases Act, regulate the importation of animals and introduce restrictions on the movement of animals in Kenya and other measures for the control of animal diseases. Importation of animals shall be done only through prescribed ports and under a licence granted by the Director of Veterinary Services. Specified certificates shall be required in respect of animals to be imported. The Rules also provide for tests, treatment and quarantine for imported cattle and other animals. Animals from Tanzania or Uganda may be imported subject to such restrictions and requirements as the Director may, from time to time, direct. No cattle, swine, sheep, goats or captive wild animal shall be moved within a restricted area, i.e. any of the areas described in the First Schedule, except under a licence issued under these Rules. Various other rules regarding the movement of animals are prescribed for purposes of the control of diseases. Other provisions concern the notif

In [12]:
# Lo descargamos porque demoró 35 minutos en correr y prefiero cortarme un testiculo antes que hacerlo de nuevo

#from google.colab import files
#df.to_excel('df.xlsx')
#files.download('df.xlsx')

#Celda aparte, es pa revisar la descarga
#df = pd.read_excel('/content/Corpus-Agro-lematizado.xlsx')
#df.head()

### Ahora que filtramos todos los datos, les asignamos un idioma, procedemos a tokenizarlo, que es ponerlo de una forma que el modelo pueda vectorizarlo para poder posteriormente predecir las aproximaciones de las consultas realizadas

In [13]:
# Tokenizar cada resumen para Word2Vec
# Esta línea crea una lista de listas donde cada sublista contiene las palabras de un resumen lematizado.
corpus_tokenizado = [str(resumen).split() for resumen in df['Resumen_lem']]

# Entrenar el modelo
# Esta línea entrena un modelo de Word2Vec utilizando el corpus tokenizado.
# - sentences: es el corpus que contiene las oraciones tokenizadas.
# - vector_size: determina la dimensionalidad de los vectores de palabras generados.
# - window: define el tamaño del contexto que se considera para las palabras (número de palabras a la izquierda y derecha de la palabra objetivo).
# - min_count: establece el número mínimo de veces que una palabra debe aparecer en el corpus para ser considerada.
# - workers: especifica el número de hilos de CPU a utilizar durante el entrenamiento para mayor velocidad.
modelo_w2v = Word2Vec(sentences=corpus_tokenizado, vector_size=100, window=5, min_count=2, workers=4)

# Guardar el modelo entrenado para usarlo después
# Esta línea guarda el modelo entrenado en un archivo con el nombre especificado.
# El modelo se podrá cargar más tarde sin necesidad de volver a entrenarlo, ahorrando tiempo y recursos.
modelo_w2v.save("modelo_word2vec.model")


In [14]:
# Función para vectorizar la consulta
def vectorizar_consulta(consulta):
    # Detectar el idioma
    try:
        idioma = detect(consulta)
    except Exception as e:
        print("Error al detectar el idioma:", e)
        return None  # O manejar el error como prefieras

    # Validar el idioma detectado
    if idioma not in ['en', 'es', 'fr']:
        print("Idioma no soportado:", idioma)
        return None  # O manejar el caso de idioma no soportado

    # Lematizar la consulta según el idioma detectado
    lematizar_consulta = Lematizar(consulta, idioma).split()
    consulta_tokenizada = [modelo_w2v.wv[consulta_token] for consulta_token in lematizar_consulta if consulta_token in modelo_w2v.wv]

    # Calcular y devolver el vector de la consulta
    return np.mean(consulta_tokenizada, axis=0)

In [15]:
# Función para vectorizar el resumen
def vectorizar_resumen(resumen):
    # Detectar el idioma
    try:
        idioma = detect(resumen)
    except Exception as e:
        print("Error al detectar el idioma:", e)
        return None  # O manejar el error como prefieras

    # Validar el idioma detectado
    if idioma not in ['en', 'es', 'fr']:
        print("Idioma no soportado:", idioma)
        return None  # O manejar el caso de idioma no soportado

    # Lematizar el resumen según el idioma detectado
    lematizar_resumen = Lematizar(resumen, idioma).split()
    resumen_tokenizado = [modelo_w2v.wv[resumen_token] for resumen_token in lematizar_resumen if resumen_token in modelo_w2v.wv]

    # Calcular y devolver el vector del resumen
    return np.mean(resumen_tokenizado, axis=0)

In [16]:
df['Vector_resumen'] = df['Resumen_lem'].apply(vectorizar_resumen)

In [17]:
def buscador(consulta, n=5):
    vector_consulta = vectorizar_consulta(consulta)
    
    # Imprimir el vector de consulta para depuración
#    print(f"Vector de consulta: {vector_consulta}")
#    print(f"Forma del vector de consulta: {vector_consulta.shape if hasattr(vector_consulta, 'shape') else 'No tiene forma'}")
    
    # Verificar si el vector de consulta es válido
    if vector_consulta is None or len(vector_consulta) == 0:
        print("El vector de consulta es vacío o nulo.")
        return "La consulta no generó un vector válido."

    # Calcular la similitud
    df['Similitud'] = df['Vector_resumen'].apply(lambda x: 1 - cosine(vector_consulta, x))
    result = df.nlargest(n, 'Similitud')

    return result[['URL de Documento', 'Resumen', 'Similitud']]


In [18]:
print(df['Vector_resumen'].apply(lambda x: (type(x), x.shape)).head().unique())


[(<class 'numpy.ndarray'>, (100,))]


In [19]:
consulta = input("Ingrese su consulta: ")
buscador(consulta, 7)

,URL de Documento,Resumen,Similitud
4,https://faolex.fao.org/docs/pdf/ken126417.pdf,"These Rules, made under the Animal Diseases Ac...",0.990125
77,https://faolex.fao.org/docs/pdf/bi-64584.pdf,The Parties have agreed to undertake the neces...,0.939114
0,https://faolex.fao.org/docs/pdf/ita32358.pdf,"The Regional Act shall apply to animal fats, s...",0.915635
19,https://faolex.fao.org/docs/pdf/oma39299.pdf,This Law consists of 2 Chapters divided into 2...,0.901961
46,https://faolex.fao.org/docs/pdf/gib182208.pdf,These Rules have the purpose to regulate the a...,0.900775
97,https://faolex.fao.org/docs/pdf/isr33332.pdf,"This Regulation, of Five Chapters and 43 Secti...",0.894714
45,https://faolex.fao.org/docs/pdf/srb208560.pdf,"This Regulation, based on the provisions of th...",0.886174
